In [3]:
import json
import time
import sqlite3

In [10]:
DB_PATH_COMFY = '/Users/lionelyip/PycharmProjects/GenImgeCrawler/data/comfyui_data.db'
TABLE_COMFY = 'comfyui_models_hash' # 确保表名正确

# Civitai 数据 (参照库)
DB_PATH_CIVITAI = '/Users/lionelyip/PycharmProjects/GenImgeCrawler/data/civitai.db'
TABLE_CIVITAI = 'civitai_file_hashes' # 确保表名正确
# ===========================================

def compare_two_sqlite_files():
    start_time = time.time()

    # ---------------------------------------------------------
    # 步骤 1: 加载 Civitai 的 Hash 和 Filename 到两个 Set
    # ---------------------------------------------------------
    print(f"[{time.strftime('%H:%M:%S')}] 正在从 Civitai 库加载参照数据...")

    civitai_hash_set = set()
    civitai_name_set = set()

    try:
        conn_civ = sqlite3.connect(DB_PATH_CIVITAI)
        cursor_civ = conn_civ.cursor()

        # 读取 hash_value 和 filename
        # 注意：这里假设你的 civitai 表里确实有 filename 这个字段
        cursor_civ.execute(f"SELECT hash_value, file_name FROM {TABLE_CIVITAI}")

        while True:
            rows = cursor_civ.fetchmany(50000)
            if not rows:
                break
            for row in rows:
                h_val = row[0]
                f_name = row[1]

                # 1. 存 Hash (大写)
                if h_val:
                    civitai_hash_set.add(h_val.strip().upper())

                # 2. 存 Filename (小写，用于兜底)
                if f_name:
                    civitai_name_set.add(f_name.strip().lower())

        conn_civ.close()
        print(f"[{time.strftime('%H:%M:%S')}] Civitai 加载完成。")
        print(f"   -> Hash池大小: {len(civitai_hash_set)}")
        print(f"   -> Name池大小: {len(civitai_name_set)}")

    except Exception as e:
        print(f"读取 Civitai 数据库出错: {e}")
        return

    print(f"[{time.strftime('%H:%M:%S')}] 开始遍历 ComfyUI 表进行匹配...")

    total_comfy = 0
    match_by_hash = 0
    match_by_name = 0

    try:
        conn_comfy = sqlite3.connect(DB_PATH_COMFY)
        cursor_comfy = conn_comfy.cursor()

        # 读取 ComfyUI 的 id, name, hash
        cursor_comfy.execute(f"SELECT id, name, hash FROM {TABLE_COMFY}")

        while True:
            row = cursor_comfy.fetchone()
            if row is None:
                break

            total_comfy += 1
            c_id = row[0]
            c_name = row[1] # 这是 ComfyUI 里的 name/filename
            c_hash = row[2]

            is_matched = False


            if c_hash:
                c_hash_upper = c_hash.strip().upper()
                if c_hash_upper in civitai_hash_set:
                    match_by_hash += 1
                    is_matched = True

            # 策略 B: 如果 Hash 没匹配上，尝试用 Name 兜底
            if not is_matched and c_name:
                c_name_lower = c_name.strip().lower()
                if c_name_lower in civitai_name_set:
                    match_by_name += 1
                    is_matched = True

            # 进度打印
            if total_comfy % 50000 == 0:
                print(f"已扫描 {total_comfy} 条...")

        conn_comfy.close()

    except Exception as e:
        print(f"读取 ComfyUI 数据库出错: {e}")
        return

    # ---------------------------------------------------------
    # 步骤 3: 输出结果
    # ---------------------------------------------------------
    end_time = time.time()
    duration = end_time - start_time

    total_matches = match_by_hash + match_by_name

    if total_comfy > 0:
        ratio = (total_matches / total_comfy) * 100
    else:
        ratio = 0

    print("\n" + "="*40)
    print(f"分析完成 (耗时 {duration:.2f} 秒)")
    print("="*40)
    print(f"ComfyUI 总记录数      : {total_comfy}")
    print("-" * 20)
    print(f"1. Hash 精确匹配数    : {match_by_hash}")
    print(f"2. Name 兜底匹配数    : {match_by_name}")
    print("-" * 20)
    print(f"总匹配成功数          : {total_matches}")
    print(f"总匹配覆盖率          : {ratio:.2f}%")
    print("="*40)

In [11]:
compare_two_sqlite_files()

[20:56:46] 正在从 Civitai 库加载参照数据...
[20:56:50] Civitai 加载完成。
   -> Hash池大小: 3095958
   -> Name池大小: 504662
[20:56:50] 开始遍历 ComfyUI 表进行匹配...
已扫描 50000 条...
已扫描 100000 条...
已扫描 150000 条...
已扫描 200000 条...
已扫描 250000 条...
已扫描 300000 条...
已扫描 350000 条...
已扫描 400000 条...
已扫描 450000 条...
已扫描 500000 条...
已扫描 550000 条...
已扫描 600000 条...
已扫描 650000 条...
已扫描 700000 条...
已扫描 750000 条...
已扫描 800000 条...
已扫描 850000 条...
已扫描 900000 条...
已扫描 950000 条...
已扫描 1000000 条...
已扫描 1050000 条...
已扫描 1100000 条...
已扫描 1150000 条...

分析完成 (耗时 4.67 秒)
ComfyUI 总记录数      : 1178313
--------------------
1. Hash 精确匹配数    : 229221
2. Name 兜底匹配数    : 21211
--------------------
总匹配成功数          : 250432
总匹配覆盖率          : 21.25%
